In [ ]:
# import numpy as np
# from scipy.stats import norm
# from statsmodels.stats.contingency_tables import mcnemar

# def coverage_test(x, alpha=0.05, power=0.80):
#     """覆盖率与基线比例比较所需样本量"""
#     p0, p1 = 0.95, x
#     z_a, z_b = norm.ppf(1 - alpha/2), norm.ppf(power)
#     return (z_a*np.sqrt(p0*(1-p0)) + z_b*np.sqrt(p1*(1-p1)))**2 / (p0 - p1)**2

# def paired_binary_test(base_ok, quant_ok):
#     """配对二值指标的 McNemar 检验"""
#     b = int(np.sum((base_ok == 1) & (quant_ok == 0)))
#     c = int(np.sum((base_ok == 0) & (quant_ok == 1)))
#     res = mcnemar(np.array([[int(np.sum((base_ok==1)&(quant_ok==1))), b],
#                             [c, int(np.sum((base_ok==0)&(quant_ok==0)))]]), exact=False)
#     return {"discordant_b": b, "discordant_c": c, "pvalue": res.pvalue}

# # 数值相对误差：报告分位数而非均值（分布重尾）
# rel = np.abs(y_quant - y_base) / (np.abs(y_base) + 1e-12)
# print("相对误差 中位数/P95：", np.median(rel), np.quantile(rel, 0.95))

In [2]:
import numpy as np
from scipy.stats import binomtest

rng = np.random.default_rng(2026)

# 合成"模型"：4 层线性网络（仅作量化误差传播的载体）
dims = [16, 64, 64, 32, 8]
Ws = [rng.normal(0, 1/np.sqrt(a), size=(a, b)) for a, b in zip(dims[:-1], dims[1:])]

def quantize_int8(W):
    """逐输出通道对称 INT8 量化，返回反量化权重"""
    s = np.abs(W).max(axis=1, keepdims=True) / 127.0
    s = np.where(s == 0, 1e-12, s)
    return np.clip(np.round(W / s), -127, 127) * s

def forward(x, weights):
    h = x
    for W in weights:
        h = np.tanh(h @ W)
    return h

Wq = [quantize_int8(W) for W in Ws]

# 合成分类任务：两个高斯类；真标签由 FP 基线在无噪特征上的输出定义
n_per = 3000
X = np.vstack([rng.normal(-0.5, 1.0, (n_per, 16)),
               rng.normal(+0.5, 1.0, (n_per, 16))])
y_label = (forward(X, Ws)[:, 0] > 0).astype(int)

# 带噪观测：比较两模型在噪声特征上的判别
Xn = X + rng.normal(0, 0.35, size=X.shape)
logit_b, logit_q = forward(Xn, Ws)[:, 0], forward(Xn, Wq)[:, 0]

# 协议第 1 项：数值一致性（相对误差分位数）
rel = np.abs(logit_q - logit_b) / np.maximum(np.abs(logit_b), 1e-6)
print(f"相对误差 中位数={np.median(rel):.5f}, P95={np.quantile(rel, 0.95):.5f}")
print(f"相关系数 = {np.corrcoef(logit_b, logit_q)[0, 1]:.5f}")

# 协议第 2 项：判别错误率（配对设计 + McNemar 不一致对）
pred_b, pred_q = (logit_b > 0).astype(int), (logit_q > 0).astype(int)
print(f"错误率: 基线={np.mean(pred_b != y_label):.4f}, 量化={np.mean(pred_q != y_label):.4f}")
b = int(((pred_b == y_label) & (pred_q != y_label)).sum())
c = int(((pred_b != y_label) & (pred_q == y_label)).sum())
print(f"不一致对 b={b}, c={c}, McNemar 精确 p={binomtest(b, b + c, 0.5).pvalue:.4f}")

相对误差 中位数=0.01506, P95=0.17644
相关系数 = 0.99988
错误率: 基线=0.1100, 量化=0.1103
不一致对 b=14, c=12, McNemar 精确 p=0.8450


In [3]:
import numpy as np

rng = np.random.default_rng(2026)
mu, sigma, n = 0.85, 0.05, 5     # 每请求质量得分 ~ N(mu, sigma^2)，子组 n=5
A2, D3, D4 = 0.577, 0.0, 2.114   # n=5 的控制图系数

def subgroups(m, shift=0.0):
    """生成 m 个子组，子组均值上移 shift 个标准差"""
    return rng.normal(mu, sigma, (m, n)) + shift * sigma

# 第一阶段：受控期 25 个子组，估计控制限
x = subgroups(25)
xbar, R = x.mean(axis=1), x.max(axis=1) - x.min(axis=1)
CL, Rbar = xbar.mean(), R.mean()
UCL, LCL = CL + A2 * Rbar, CL - A2 * Rbar
print(f"CL={CL:.4f}, UCL={UCL:.4f}, LCL={LCL:.4f}, sigma_hat={Rbar/2.326:.4f}")

# 受控期验证：另取 100 个子组，统计误报
xbar_ic = subgroups(100).mean(axis=1)
print(f"受控期误报: {int(((xbar_ic > UCL) | (xbar_ic < LCL)).sum())}/100 子组")

# 第二阶段：注入持续漂移，观察首次报警子组
for delta in (0.5, 1.0, 2.0):
    xbar_oc = subgroups(30, shift=delta).mean(axis=1)
    hits = np.where((xbar_oc > UCL) | (xbar_oc < LCL))[0]
    print(f"漂移 {delta}sigma: 首次报警子组 = "
          f"{hits[0] + 1 if len(hits) else None}, 30 个子组报警 {len(hits)} 次")

CL=0.8531, UCL=0.9170, LCL=0.7892, sigma_hat=0.0476
受控期误报: 0/100 子组
漂移 0.5sigma: 首次报警子组 = 17, 30 个子组报警 1 次
漂移 1.0sigma: 首次报警子组 = 2, 30 个子组报警 8 次
漂移 2.0sigma: 首次报警子组 = 2, 30 个子组报警 27 次


In [4]:
import numpy as np

rng = np.random.default_rng(2026)
LAM, K = 0.2, 3.0

def arl(chart, delta, max_t=20000, reps=20000):
    """delta=0 时返回 ARL0，否则返回漂移检出延迟"""
    stops = []
    for _ in range(reps):
        z, t_hit = 0.0, max_t
        for t in range(1, max_t + 1):
            x = rng.normal(delta, 1.0)
            if chart == "shewhart":
                hit = abs(x) > K
            else:
                z = LAM * x + (1 - LAM) * z
                lim = K * np.sqrt(LAM / (2 - LAM) * (1 - (1 - LAM) ** (2 * t)))
                hit = abs(z) > lim
            if hit:
                t_hit = t
                break
        stops.append(t_hit)
    s = np.array(stops, float)
    return s.mean(), np.quantile(s, [0.5, 0.95])

for delta in (0.0, 0.5, 1.0):
    a_sh, _ = arl("shewhart", delta)
    a_ew, _ = arl("ewma", delta)
    print(f"漂移 {delta:.1f}sigma: Shewhart ARL={a_sh:.1f}, EWMA(0.2) ARL={a_ew:.1f}")

漂移 0.0sigma: Shewhart ARL=371.8, EWMA(0.2) ARL=545.1
漂移 0.5sigma: Shewhart ARL=154.4, EWMA(0.2) ARL=42.6
漂移 1.0sigma: Shewhart ARL=43.5, EWMA(0.2) ARL=9.8


In [5]:
import numpy as np
from scipy.stats import norm

def n_prop(p1, p2, alpha=0.05, power=0.80, deff=1.0):
    z_a, z_b = norm.ppf(1 - alpha/2), norm.ppf(power)
    return int(np.ceil((z_a + z_b)**2 * (p1*(1-p1) + p2*(1-p2)) / (p2-p1)**2 * deff))

def mde_prop(p1, n, alpha=0.05, power=0.80):
    """给定样本量反解可检出的最小差异（数值求解）"""
    from scipy.optimize import brentq
    f = lambda d: n_prop(p1, p1 + d, alpha, power) - n
    return brentq(f, 1e-4, 0.5 - 1e-4)

print(n_prop(0.70, 0.73))            # 约 3550（按独立单元）
print(n_prop(0.70, 0.73, deff=2.9))  # 聚类随机化下约 10300
print(round(mde_prop(0.70, 3000), 4))  # 每组 3000 时的最小可检出差异

3551
10296
0.0326


In [ ]:
# import numpy as np

# def cuped_adjust(y, x):
#     """y: 实验期指标；x: 实验前同口径指标（同一单元，缺失可用均值填）"""
#     x = np.asarray(x, float); y = np.asarray(y, float)
#     theta = np.cov(x, y, ddof=1)[0, 1] / np.var(x, ddof=1)
#     return y - theta * (x - x.mean()), theta

# y_adj, theta = cuped_adjust(y_treatment, x_treatment)
# print("theta =", theta, "方差缩减比例 =", 1 - np.var(y_adj, ddof=1) / np.var(y_treatment, ddof=1))

In [7]:
import numpy as np
from scipy.stats import norm

rng = np.random.default_rng(2026)
N, LOOKS, ALPHA, REPS = 1000, 5, 0.05, 20000

def sim(strategy):
    rejections = 0
    for _ in range(REPS):
        a = rng.normal(0, 1, N)     # 处理组（零效应）
        b = rng.normal(0, 1, N)     # 对照组
        rejected = False
        for k in range(1, LOOKS + 1):
            n = k * N // LOOKS
            z = (a[:n].mean() - b[:n].mean()) / np.sqrt(2.0 / n)
            if strategy == "fixed":
                if k == LOOKS:
                    rejected = abs(z) > 1.96
            elif strategy == "peek":
                if abs(z) > 1.96:
                    rejected = True
                    break
            elif strategy == "obf":
                c = norm.ppf(1 - ALPHA / 2) / np.sqrt(k / LOOKS)
                if abs(z) > c:
                    rejected = True
                    break
        rejections += rejected
    return rejections / REPS

for s in ("fixed", "peek", "obf"):
    print(f"{s:>6}: 零效应下拒绝比例 = {sim(s):.4f}")

 fixed: 零效应下拒绝比例 = 0.0493
  peek: 零效应下拒绝比例 = 0.1442
   obf: 零效应下拒绝比例 = 0.0607


In [8]:
from scipy.stats import chisquare
obs = [50231, 49812]          # 两组的实际样本量
exp = [sum(obs) * 0.5] * 2    # 预设 50/50
print(chisquare(f_obs=obs, f_exp=exp))

Power_divergenceResult(statistic=1.7548554121727657, pvalue=0.18526750191209126)
